<a href="https://colab.research.google.com/github/castr1x/ML_an.nykoliuk_3-12/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%9D%D0%B8%D0%BA%D0%BE%D0%BB%D1%8E%D0%BA%D0%B0_%D0%A4%D0%86%D0%A2_3_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** _Николюк Андрій Денисович_  
**Група:** _____ФІТ 3-12____________________  
**Дата:** ______22.11.2026__________________

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.9
NumPy: 2.2.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
# TODO 2.1 — Дослідження набору даних
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
n_ratings = len(ratings_df)
sparsity = 1 - n_ratings / (n_users * n_items)

print('Типи колонок:')
print(ratings_df.dtypes, end='\n\n')
print(f'Кількість унікальних користувачів: {n_users}')
print(f'Кількість унікальних фільмів:      {n_items}')
print(f'Кількість оцінок:                  {n_ratings}')
print(f'Розрідженість матриці user-item:   {sparsity:.2%}', end='\n\n')

print('Статистика rating:')
print(f'  min  = {ratings_df["rating"].min()}')
print(f'  max  = {ratings_df["rating"].max()}')
print(f'  mean = {ratings_df["rating"].mean():.4f}')
print(f'  std  = {ratings_df["rating"].std():.4f}', end='\n\n')

print('Пропущені значення по колонках:')
print(ratings_df.isna().sum(), end='\n\n')

print('Розподіл оцінок:')
rating_dist = (ratings_df['rating']
               .value_counts()
               .sort_index()
               .rename('count')
               .to_frame())
rating_dist['share, %'] = (rating_dist['count'] / n_ratings * 100).round(2)
display(rating_dist)

user_counts = ratings_df.groupby('user_id').size()
item_counts = ratings_df.groupby('item_id').size()
print(f'Оцінок на користувача: min={user_counts.min()}, median={user_counts.median():.0f}, max={user_counts.max()}')
print(f'Оцінок на фільм:       min={item_counts.min()}, median={item_counts.median():.0f}, max={item_counts.max()}')

Типи колонок:
user_id       object
item_id       object
rating       float64
timestamp     object
dtype: object

Кількість унікальних користувачів: 943
Кількість унікальних фільмів:      1682
Кількість оцінок:                  100000
Розрідженість матриці user-item:   93.70%

Статистика rating:
  min  = 1.0
  max  = 5.0
  mean = 3.5299
  std  = 1.1257

Пропущені значення по колонках:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:


,count,"share, %"
rating,,
1.0,6110,6.11
2.0,11370,11.37
3.0,27145,27.15
4.0,34174,34.17
5.0,21201,21.20


Оцінок на користувача: min=20, median=65, max=737
Оцінок на фільм:       min=1, median=27, max=583


**Короткі спостереження щодо даних:**

Набір містить 100 000 оцінок від 943 користувачів для 1682 фільмів; пропущених значень немає, ID мають рядковий тип (raw ID). Матриця «користувач–фільм» дуже розріджена (≈93.7% порожніх клітинок), кожен користувач оцінив щонайменше 20 фільмів (медіана 65), а от фільми мають від 1 до 583 оцінок, тобто є «довгий хвіст» рідко оцінених фільмів. Оцінки цілі від 1 до 5 і зміщені у бік позитивних: середнє 3.53, найчастіша оцінка — 4 (34%), а низьких оцінок 1–2 лише ≈17%.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
# TODO 3.1 — Розбиття 80% / 20%
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)

print('train_df:', train_df.shape)
print('test_df: ', test_df.shape)

train_df: (80000, 4)
test_df:  (20000, 4)


In [ ]:
# TODO 3.2 — Об’єкти для Surprise
reader = Reader(rating_scale=data.reader.rating_scale)
print('Шкала оцінок:', reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print('Навчальна вибірка:', len(train_data.raw_ratings), 'оцінок')
print('Тестова вибірка:  ', len(testset), 'оцінок')
print('Приклад тестового кортежу:', testset[0])

# Скільки користувачів/фільмів тестової вибірки відсутні у навчальній
unknown_users = set(test_df['user_id']) - set(train_df['user_id'])
unknown_items = set(test_df['item_id']) - set(train_df['item_id'])
print('Нових користувачів у test:', len(unknown_users))
print('Нових фільмів у test:     ', len(unknown_items))

Шкала оцінок: (1, 5)


Навчальна вибірка: 80000 оцінок
Тестова вибірка:   20000 оцінок
Приклад тестового кортежу: ('877', '381', 4.0)


Нових користувачів у test: 0
Нових фільмів у test:      29


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

- **SVD:** `n_factors` — розмірність прихованого простору (50 vs 100): більше факторів дає гнучкість, але на 80 тис. оцінок легко перенавчитися; `lr_all` — крок SGD (0.005 — стандартне значення Surprise, 0.01 — швидша збіжність за ті самі 20 епох); `reg_all` — L2-регуляризація (0.02 — стандарт, 0.05 — сильніша, корисна для розріджених даних).
- **KNNBasic:** `k` — максимальна кількість сусідів (20/40: компроміс між шумом і згладжуванням); `min_k` — мінімум сусідів, інакше прогноз = глобальне середнє (1/5); `sim_options` — метрика схожості (`cosine`/`pearson`) та підхід user-based чи item-based.

Сітку залишено стартовою з прикладу: вона невелика (8 і 16 комбінацій × 3 фолди), охоплює значення за замовчуванням та «сусідні» до них і виконується за прийнятний час.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
# TODO 5.1 — GridSearchCV для SVD
grid_search_svd = GridSearchCV(
    SVD, param_grid_svd,
    measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)

print('SVD: перебрано комбінацій —', len(grid_search_svd.cv_results['params']))

SVD: перебрано комбінацій — 8


In [ ]:
# TODO 5.2 — GridSearchCV для KNNBasic
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn,
    measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)

print('KNNBasic: перебрано комбінацій —', len(grid_search_knn.cv_results['params']))

KNNBasic: перебрано комбінацій — 16


In [ ]:
# TODO 5.3 — Найкращі результати кросвалідації
def cv_summary(gs, name):
    res = pd.DataFrame(gs.cv_results)
    best_idx = res['rank_test_rmse'].idxmin()
    print(f'=== {name} ===')
    print(f"Найкращий CV RMSE: {gs.best_score['rmse']:.4f}")
    print(f"CV MAE (для тієї ж комбінації): {res.loc[best_idx, 'mean_test_mae']:.4f}")
    print(f"Найкращий CV MAE (окремо):      {gs.best_score['mae']:.4f}")
    print(f"Параметри за RMSE: {gs.best_params['rmse']}")
    print(f"Параметри за MAE:  {gs.best_params['mae']}", end='\n\n')
    return res

svd_cv = cv_summary(grid_search_svd, 'SVD')
knn_cv = cv_summary(grid_search_knn, 'KNNBasic')

# Повні таблиці CV (топ-5 за RMSE)
cols = ['params', 'mean_test_rmse', 'std_test_rmse', 'mean_test_mae', 'rank_test_rmse']
print('SVD — топ-5 комбінацій:')
display(svd_cv.sort_values('rank_test_rmse')[cols].head(5))
print('KNNBasic — топ-5 комбінацій:')
display(knn_cv.sort_values('rank_test_rmse')[cols].head(5))

=== SVD ===
Найкращий CV RMSE: 0.9409
CV MAE (для тієї ж комбінації): 0.7425
Найкращий CV MAE (окремо):      0.7425
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Параметри за MAE:  {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

=== KNNBasic ===
Найкращий CV RMSE: 1.0275
CV MAE (для тієї ж комбінації): 0.8158
Найкращий CV MAE (окремо):      0.8147
Параметри за RMSE: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}
Параметри за MAE:  {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}

SVD — топ-5 комбінацій:


,params,mean_test_rmse,std_test_rmse,mean_test_mae,rank_test_rmse
3,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0...",0.940937,0.002788,0.742486,1
7,"{'n_factors': 100, 'lr_all': 0.01, 'reg_all': ...",0.943183,0.001805,0.744246,2
1,"{'n_factors': 50, 'lr_all': 0.005, 'reg_all': ...",0.947124,0.002564,0.750075,3
5,"{'n_factors': 100, 'lr_all': 0.005, 'reg_all':...",0.948084,0.002097,0.750477,4
0,"{'n_factors': 50, 'lr_all': 0.005, 'reg_all': ...",0.950341,0.002081,0.751053,5


KNNBasic — топ-5 комбінацій:


,params,mean_test_rmse,std_test_rmse,mean_test_mae,rank_test_rmse
13,"{'k': 40, 'min_k': 5, 'sim_options': {'name': ...",1.027481,0.003675,0.815753,1
9,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ...",1.027520,0.004036,0.814748,2
15,"{'k': 40, 'min_k': 5, 'sim_options': {'name': ...",1.028301,0.004049,0.818648,3
11,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ...",1.031201,0.003345,0.818090,4
7,"{'k': 20, 'min_k': 5, 'sim_options': {'name': ...",1.034499,0.004570,0.823821,5


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9409 | 0.7425 | `n_factors=50, lr_all=0.01, reg_all=0.05` |
| KNNBasic | 1.0275 | 0.8158 | `k=40, min_k=5, sim=cosine, user_based=True` |

SVD суттєво точніший за KNNBasic: різниця в CV RMSE ≈0.087 (≈8.4%), у MAE ≈0.073. Для SVD краще працюють менша кількість факторів і сильніша регуляризація з більшим кроком — модель з меншою складністю краще узагальнює на розріджених даних. Для KNNBasic найкращим є user-based підхід з косинусною схожістю та більшою кількістю сусідів (k=40); `min_k` майже не впливає (1.0275 vs 1.0275), а за MAE найкращим виявився `min_k=1` (0.8147). Стандартні відхилення між фолдами малі (0.002–0.004), тож різниця між моделями стабільна.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# TODO 6.1 — Найкращі параметри за CV RMSE
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']
print('best_svd_params:', best_svd_params)
print('best_knn_params:', best_knn_params)

# TODO 6.2 — Повна навчальна вибірка
trainset = train_data.build_full_trainset()
print(f'trainset: {trainset.n_users} користувачів, {trainset.n_items} фільмів, '
      f'{trainset.n_ratings} оцінок')

# TODO 6.3 — Навчання остаточних моделей
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params, verbose=False)
best_knn.fit(trainset)

print('Моделі навчено.')

best_svd_params: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
best_knn_params: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}


trainset: 943 користувачів, 1653 фільмів, 80000 оцінок


Моделі навчено.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# TODO 7.1 — Передбачення на тестовій вибірці
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Метрики
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD:      RMSE = {svd_rmse:.4f}, MAE = {svd_mae:.4f}')
print(f'KNNBasic: RMSE = {knn_rmse:.4f}, MAE = {knn_mae:.4f}')

# Приклади передбачень
print()
for p in svd_predictions[:5]:
    print(f'user={p.uid:>4}  item={p.iid:>5}  true={p.r_ui}  SVD est={p.est:.3f}')

SVD:      RMSE = 0.9192, MAE = 0.7234
KNNBasic: RMSE = 1.0117, MAE = 0.7993

user= 877  item=  381  true=4.0  SVD est=3.665
user= 815  item=  602  true=3.0  SVD est=3.300
user=  94  item=  431  true=4.0  SVD est=3.496
user= 416  item=  875  true=2.0  SVD est=3.008
user= 500  item=  182  true=2.0  SVD est=4.100


In [ ]:
# TODO 7.3 — Порівняльна таблиця
svd_cv_mae = svd_cv.loc[svd_cv['rank_test_rmse'].idxmin(), 'mean_test_mae']
knn_cv_mae = knn_cv.loc[knn_cv['rank_test_rmse'].idxmin(), 'mean_test_mae']

results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae],
    'CV RMSE': [grid_search_svd.best_score['rmse'], grid_search_knn.best_score['rmse']],
    'CV MAE': [svd_cv_mae, knn_cv_mae],
}).sort_values('RMSE').reset_index(drop=True)

results['RMSE - CV RMSE'] = results['RMSE'] - results['CV RMSE']
display(results.round(4))

,Model,RMSE,MAE,CV RMSE,CV MAE,RMSE - CV RMSE
0,SVD,0.9192,0.7234,0.9409,0.7425,-0.0218
1,KNNBasic,1.0117,0.7993,1.0275,0.8158,-0.0158


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0117 | 0.7993 |

**Порівняння з CV:** Оцінки на тестовій вибірці дещо кращі, ніж на кросвалідації (SVD: 0.9192 vs 0.9409; KNNBasic: 1.0117 vs 1.0275). Це очікувано: під час CV моделі навчалися лише на 2/3 навчальної частини (≈53 тис. оцінок), а фінальні моделі — на всіх 80 тис., тому вони мають більше інформації про користувачів і фільми. Різниця невелика (0.016–0.022), порядок моделей зберігся, отже перенавчання на етапі підбору гіперпараметрів немає, а CV-оцінка є надійною (трохи песимістичною).

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
user_id = '196'

# TODO 8.1 — Найкраща модель за CV RMSE
cv_scores = {
    'SVD': (grid_search_svd.best_score['rmse'], best_svd),
    'KNNBasic': (grid_search_knn.best_score['rmse'], best_knn),
}
best_model_name = min(cv_scores, key=lambda name: cv_scores[name][0])
best_model = cv_scores[best_model_name][1]
print(f'Обрана модель: {best_model_name} (CV RMSE = {cv_scores[best_model_name][0]:.4f})')

# Назви фільмів (файл u.item з набору ml-100k) — для наочності
import os
from surprise.builtin_datasets import get_dataset_dir

def load_movie_titles():
    item_file = os.path.join(get_dataset_dir(), 'ml-100k', 'ml-100k', 'u.item')
    titles = {}
    try:
        with open(item_file, encoding='ISO-8859-1') as f:
            for line in f:
                parts = line.split('|')
                titles[parts[0]] = parts[1]
    except OSError:
        pass
    return titles

movie_titles = load_movie_titles()


def get_top_n(model, trainset, user_id, n=10):
    # TODO 8.2 — Перевірка наявності користувача у trainset
    try:
        inner_uid = trainset.to_inner_uid(user_id)
    except ValueError:
        print(f'Користувача {user_id} немає у trainset (холодний старт). '
              f'Показуємо найпопулярніші фільми з високим середнім рейтингом.')
        stats = train_df.groupby('item_id')['rating'].agg(['mean', 'count'])
        top = (stats[stats['count'] >= 50]
               .sort_values('mean', ascending=False)
               .head(n)
               .reset_index()
               .rename(columns={'mean': 'predicted_rating'}))
        return top[['item_id', 'predicted_rating']]

    # TODO 8.3 — Неоцінені фільми
    rated_inner = {iid for iid, _ in trainset.ur[inner_uid]}
    unrated_raw = [trainset.to_raw_iid(iid)
                   for iid in trainset.all_items() if iid not in rated_inner]
    print(f'Користувач {user_id}: оцінено у train — {len(rated_inner)}, '
          f'кандидатів для рекомендації — {len(unrated_raw)}')

    # TODO 8.4 — Прогноз і топ-n
    preds = [(iid, model.predict(uid=user_id, iid=iid).est) for iid in unrated_raw]
    top = (pd.DataFrame(preds, columns=['item_id', 'predicted_rating'])
           .sort_values('predicted_rating', ascending=False)
           .head(n)
           .reset_index(drop=True))
    return top


top10 = get_top_n(best_model, trainset, user_id, n=10)
top10['title'] = top10['item_id'].map(movie_titles)
top10.index = top10.index + 1
display(top10)

# Перевірка: жоден рекомендований фільм не оцінений користувачем у train
already_rated = set(train_df.loc[train_df['user_id'] == user_id, 'item_id'])
print('Перетин з уже оціненими фільмами:', set(top10['item_id']) & already_rated or 'немає')

Обрана модель: SVD (CV RMSE = 0.9409)
Користувач 196: оцінено у train — 29, кандидатів для рекомендації — 1624


,item_id,predicted_rating,title
1,320,4.490358,Paradise Lost: The Child Murders at Robin Hood...
2,64,4.470680,"Shawshank Redemption, The (1994)"
3,480,4.413710,North by Northwest (1959)
4,313,4.401920,Titanic (1997)
5,513,4.399624,"Third Man, The (1949)"
6,483,4.398092,Casablanca (1942)
7,318,4.392102,Schindler's List (1993)
8,656,4.385379,M (1931)
9,22,4.377388,Braveheart (1995)
10,272,4.376010,Good Will Hunting (1997)


Перетин з уже оціненими фільмами: немає


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

1. **Обрана модель — SVD**, бо вона має найменший CV RMSE (0.9409 проти 1.0275 у KNNBasic) і також кращий CV MAE (0.7425 проти 0.8158).
2. **Найкращі гіперпараметри:** SVD — `n_factors=50, lr_all=0.01, reg_all=0.05`; KNNBasic — `k=40, min_k=5, sim_options={'name': 'cosine', 'user_based': True}`.
3. **Тестові метрики:** SVD — RMSE 0.9192, MAE 0.7234; KNNBasic — RMSE 1.0117, MAE 0.7993.
4. **CV vs тест:** тестові помилки на 0.016–0.022 менші, бо фінальні моделі навчені на більшому обсязі даних; ранжування моделей не змінилося.
5. **Обмеження:** проблема холодного старту — для нових користувачів моделі не можуть дати персональних рекомендацій (передбачено fallback на популярні фільми), а 29 фільмів тестової вибірки взагалі відсутні в навчальній, тож для них прогноз зводиться до середніх значень. Рекомендації базуються лише на оцінках, без жанрів, часу та контексту; моделі схильні піднімати рідко оцінені фільми з кількома високими оцінками (напр. *Paradise Lost* на 1-му місці для користувача 196), а RMSE не оцінює якість саме ранжування топ-10.

**Висновок:** У роботі на наборі MovieLens 100K (100 000 оцінок, 943 користувачі, 1682 фільми, розрідженість 93.7%) побудовано та порівняно дві рекомендаційні моделі бібліотеки Surprise. Гіперпараметри підібрано за допомогою GridSearchCV (cv=3) лише на навчальній частині (80%). Модель SVD з 50 прихованими факторами показала найкращий результат як на кросвалідації (RMSE 0.9409), так і на незалежній тестовій вибірці (RMSE 0.9192, MAE 0.7234), що на ≈9% краще за KNNBasic (RMSE 1.0117, MAE 0.7993). Перевага SVD пояснюється тим, що матрична факторизація узагальнює інформацію з усієї матриці оцінок, тоді як KNN спирається лише на прямі збіги між користувачами, яких мало при високій розрідженості. Близькість CV- і тестових метрик свідчить про відсутність перенавчання при підборі параметрів. За допомогою SVD сформовано топ-10 рекомендацій для користувача 196 (прогнозовані оцінки 4.38–4.49) — серед них класичні високо оцінені фільми (*The Shawshank Redemption*, *Casablanca*, *Schindler's List*), причому жоден із уже переглянутих фільмів не потрапив у список. Основним обмеженням залишається холодний старт для нових користувачів і фільмів.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**